In [103]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [112]:
ref_size="100Mb"

In [131]:
df = pd.read_csv(ref_size + "/search_efficiency_0.06.log", sep = "\t")
df.columns = ["Time (s)", "Memory (kb)", "exit_code", "type", "adapt_cutoff", "errors", "more"]
more = df['more'].str.split('--',expand=True)

df.drop(["exit_code", "type", "more"], axis = 1, inplace = True)
df.head()

,Time (s),Memory (kb),adapt_cutoff,errors
0,3.96,1487004,0.5,3
1,131.32,287716,0.5,3
2,5.40,2441628,0.5,3
3,61.97,379976,0.5,3
4,64.55,366892,0.5,3


In [132]:
to_remove = []
for col in more.columns:
    if (len(np.unique(more[col])) == 1):
        to_remove.append(col)

to_remove.append(14)
more.drop(to_remove, axis = 1, inplace = True)
more.columns = ["exec-path", "seg-count", "cart-max-capacity", "threads", "output-path"]
more["exec-path"] = more["exec-path"].str.removeprefix("/group/ag_abi/evelina/valik/")
more["exec-path"] = more["exec-path"].str.removesuffix("/bin/dream-stellar search ")

for col in ["seg-count", "cart-max-capacity", "threads"]:
    more[col] = more[col].str.removeprefix(col)

col = "output-path"
more[col] = more[col].str.removeprefix("output /buffer/ag_abi/evelina/markov/")
more[["ref-size", "type", "er", "output-type"]] = more[col].str.split('/',expand=True)
more.drop(["output-path", "output-type", "type"], axis = 1, inplace = True)

col = "er"
more[col] = more[col].str.removeprefix("efficiency_")

more.head()

,exec-path,seg-count,cart-max-capacity,threads,ref-size,er
0,build,100,500,32,100Mb,0.06
1,build_io,100,500,1,100Mb,0.06
2,build_io,100,500,32,100Mb,0.06
3,build_io,100,500,2,100Mb,0.06
4,build_io,100,500,2,100Mb,0.06


In [133]:
df = df.merge(more, how = "outer", left_index = True, right_index = True)
df = df.astype({'Memory (kb)': 'int64', "seg-count": 'int32', "cart-max-capacity": 'int32', "threads": 'int8'})
df.head()

,Time (s),Memory (kb),adapt_cutoff,errors,exec-path,seg-count,cart-max-capacity,threads,ref-size,er
0,3.96,1487004,0.5,3,build,100,500,32,100Mb,0.06
1,131.32,287716,0.5,3,build_io,100,500,1,100Mb,0.06
2,5.40,2441628,0.5,3,build_io,100,500,32,100Mb,0.06
3,61.97,379976,0.5,3,build_io,100,500,2,100Mb,0.06
4,64.55,366892,0.5,3,build_io,100,500,2,100Mb,0.06


In [134]:
bits_in_bt = "1377675"
ibf_size = 169 * 1024 * 1024

kmer_size = 18
shape_weight = 12
window_size = 20

bins = 1024
database_size = 104857600
query_size = 1048610
min_len = 50

In [139]:
df = df[df["seg-count"]>100]
df = df[df["cart-max-capacity"] > 1]
df.head()

,Time (s),Memory (kb),adapt_cutoff,errors,exec-path,seg-count,cart-max-capacity,threads,ref-size,er
11,61.53,314900,0.5,3,build_io,3495,500,2,100Mb,0.06
12,30.45,425932,0.5,3,build_io,3495,500,4,100Mb,0.06
13,11.30,473280,0.5,3,build,3495,500,2,100Mb,0.06
14,6.23,484580,0.5,3,build,3495,500,4,100Mb,0.06
16,71.32,478236,0.5,3,build,3495,10,1,100Mb,0.06


In [140]:
1048610 / 3495

300.0314735336195

In [144]:
build = df[df["exec-path"] == "build"].copy()
build.drop(["adapt_cutoff", "errors", "exec-path", "seg-count", "ref-size", "er"], axis = 1, inplace = True)
io = df[df["exec-path"] == "build_io"].copy()
build.drop(["adapt_cutoff", "errors", "exec-path", "seg-count", "ref-size", "er"], axis = 1, inplace = True)

In [151]:
build["memory-model"] = (build["Memory (kb)"] - (ibf_size + database_size + query_size) / 1024).astype(int)
build["memory-model"] = build["memory-model"] - build["cart-max-capacity"] * query_size / df["seg-count"].iloc[0] / 1024
build["memory-model-error"] = np.round((1 - build["memory-model"] / build["Memory (kb)"]) * 100, 2)
build.head() 

,Time (s),Memory (kb),cart-max-capacity,threads,memory-model,memory-model-error
13,11.30,473280,500,2,196652.500257,58.45
14,6.23,484580,500,4,207952.500257,57.09
16,71.32,478236,10,1,201752.070005,57.81
17,25.19,469852,100,1,193341.700051,58.85
18,21.13,470700,1000,1,193926.000514,58.80


In [154]:
data = io 
data["memory-model"] = (data["Memory (kb)"] - (ibf_size + query_size) / 1024).astype(int)
data["memory-model"] = data["memory-model"] - data["cart-max-capacity"] * query_size / df["seg-count"].iloc[0] / 1024
data["memory-model-error"] = np.round((1 - data["memory-model"] / data["Memory (kb)"]) * 100, 2)
data.head() 

,Time (s),Memory (kb),adapt_cutoff,errors,exec-path,seg-count,cart-max-capacity,threads,ref-size,er,memory-model,memory-model-error
11,61.53,314900,0.5,3,build_io,3495,500,2,100Mb,0.06,140672.500257,55.33
12,30.45,425932,0.5,3,build_io,3495,500,4,100Mb,0.06,251704.500257,40.91
28,1481.19,303344,0.5,3,build_io,3495,10,1,100Mb,0.06,129260.070005,57.39
29,247.04,274844,0.5,3,build_io,3495,100,1,100Mb,0.06,100733.700051,63.35
30,131.97,259940,0.5,3,build_io,3495,1000,1,100Mb,0.06,85566.000514,67.08
